# Cross-lithology comparison

Everything in this study that compares augen gneiss with psammitic schist, or
fits a model to both at once. The per-specimen fields for each rock are in
`Tensile_augen_gneiss.ipynb` and `Tensile_psammitic_schist.ipynb`.

**This notebook contains no function definitions.** The sections below call
`tools/analysis/`, the same package the lithology notebooks use, so a model
fitted here and a field plotted there cannot disagree about the rock.

## 1. Environment

In [1]:
%matplotlib inline
# Inline is the default under ipykernel, but stating it makes the stored
# outputs of a batch `nbconvert --execute` run independent of that default.

import numpy as np, pandas as pd, matplotlib

from tools import lithology, analysis, output_dirs
from tools.plot_style import apply_plot_style

output_dirs.ensure(*output_dirs.ALL)   # a clean checkout carries no outputs/

TICKS = apply_plot_style()

GNEISS, SCHIST = lithology.AUGEN_GNEISS, lithology.PSAMMITIC_SCHIST
print('numpy', np.__version__, '| pandas', pd.__version__,
      '| matplotlib', matplotlib.__version__)
for r in (GNEISS, SCHIST):
    print(f'{r.display_name:18s} samples {r.sample_ids}  '
          f'spacing {r.spacing_m * 1e3:.1f} mm')

numpy 2.3.0 | pandas 2.3.0 | matplotlib 3.10.3
Augen gneiss       samples (1, 2, 3, 4, 5, 6, 7)  spacing 10.0 mm
Psammitic schist   samples (8, 9, 10, 11, 12, 13, 14)  spacing 2.0 mm


## 2. Mixed-mode fraction against k_max

The mixed-mode area fraction as the local damage cap is swept, computed
separately for the gneiss and schist specimen groups.

In [2]:
analysis.kmax_sweep.main()

Sweep table saved to outputs/tables/kmax_softening_sweep.csv
  specimen-to-specimen SD: gneiss 0.03-0.08, schist 0.15-0.17 (mean level 0.05 and 0.13)
Best kC_max for Augen gneiss      : 0.900
Best kT_max for Augen gneiss      : 0.315
Best kC_max for Psammitic schist  : 0.100
Best kT_max for Psammitic schist  : 0.035

Metric used for selection: mixed / failed_total
Selection rule: smallest k within 98.0% of peak metric
Model: local anisotropic energy-driven damage softening
[WARN] Augen gneiss curve is still near peak at the upper scan bound.


Figure saved to outputs/figures/kmax_local_anisotropic_mixed_fraction.pdf
Figure saved to manuscript/kmax_local_anisotropic_mixed_fraction.pdf


<Figure size 2040x1320 with 1 Axes>

## 3. Local Mohr-Coulomb classification

The local Mohr-Coulomb failure-mode classification over all fourteen
specimens, across the spacing and heterogeneity grid.

In [3]:
analysis.mohr_coulomb_local.main()



==== SAMPLE 1 ====
Rock type : Augen gneiss
kC_max used: 0.050
kT_max used: 0.018

--- SPACING EFFECT (heterogeneity fixed) ---
Spacing = 0.0000 m -> ΔFailure:      0  (  0.00 %)
Spacing = 0.0010 m -> ΔFailure:      0  (  0.00 %)
Spacing = 0.0020 m -> ΔFailure:      0  (  0.00 %)
Spacing = 0.0030 m -> ΔFailure:      0  (  0.00 %)
Spacing = 0.0040 m -> ΔFailure:      0  (  0.00 %)

--- MATERIAL HETEROGENEITY EFFECT (spacing fixed) ---
Heterogeneity = 0.00 -> ΔFailure:      0  (  0.00 %)
Heterogeneity = 0.10 -> ΔFailure:      0  (  0.00 %)
Heterogeneity = 0.20 -> ΔFailure:      0  (  0.00 %)
Heterogeneity = 0.30 -> ΔFailure:      0  (  0.00 %)
Heterogeneity = 0.40 -> ΔFailure:     48  (  2.56 %)


==== SAMPLE 2 ====
Rock type : Augen gneiss
kC_max used: 0.050
kT_max used: 0.018

--- SPACING EFFECT (heterogeneity fixed) ---
Spacing = 0.0000 m -> ΔFailure:      0  (  0.00 %)
Spacing = 0.0010 m -> ΔFailure:      0  (  0.00 %)
Spacing = 0.0020 m -> ΔFailure:      0  (  0.00 %)
Spacing = 0.

## 4. Metric export

The formatted spreadsheet of manuscript values.

In [4]:
analysis.metric_export.main()

Excel file saved to: rock_strength_metrics.xlsx


---

# Consolidated cross-lithology diagnostics

Strength anisotropy in both loading modes, the asymmetric strength envelope and
its fit, the fabric-resolved tractions, failure statistics counted from the
fields rather than carried as literals, sensitivity to spacing and disorder, the
softening cap, fracture-trace validation against a loading-direction null, the
supplementary bounds, and a check that every graphic the manuscript cites
exists.

Per-lithology diagnostics stay in the two lithology notebooks. Every cell here
is self-contained.

In [5]:
from pathlib import Path
import warnings
import numpy as np, pandas as pd

warnings.filterwarnings("ignore")
REPO = Path.cwd()          # notebooks run from the repository root, so
                           # tools/ is importable without touching sys.path

from tools.figure_scripts import run
from tools import fabric_tractions as ft
from tools.data_io import load_specimen_table, load_replicate_table

assert len(ft.field_files()) == 14, "run both lithology notebooks first"
print(f"{len(load_specimen_table())} specimens, "
      f"{len(load_replicate_table())} replicate measurements, 14 fields")

14 specimens, 52 replicate measurements, 14 fields


In [6]:
# Strength anisotropy, both loading modes, from the replicate table
from tools import strength_anisotropy as sa
print(sa.per_angle_means().to_string(index=False))
print()
r = sa.anisotropy_ratios().set_index("lithology")
for lit in r.index:
    print(f"{lit:17s} tensile {r.loc[lit,'tensile_anisotropy']:.2f}  "
          f"compressive {r.loc[lit,'compressive_anisotropy']:.2f}  "
          f"separation {abs(r.loc[lit,'relative_separation_pct']):.0f}%")

       lithology  angle_deg  tensile_MPa  compressive_MPa  n_replicates
    Augen gneiss          0    10.470000        57.472000             5
    Augen gneiss         15    10.026000        50.884000             5
    Augen gneiss         30     9.794000        42.114000             5
    Augen gneiss         45     9.147500        48.567500             4
    Augen gneiss         60     7.910000        48.950000             3
    Augen gneiss         75     7.630000        61.342500             4
    Augen gneiss         90     8.000000        67.462500             4
Psammitic schist          0     9.570000        42.506667             3
Psammitic schist         15     8.436667        40.293333             3
Psammitic schist         30     8.106667        28.026667             3
Psammitic schist         45     6.860000        18.983333             3
Psammitic schist         60     5.900000        36.830000             3
Psammitic schist         75     3.717500        39.602500       

In [7]:
# Asymmetric strength envelope, and the cosine law fitted to the same data.
# End members are measured, so only the depth and position of the weakening
# are fitted; intervals are profile-likelihood ranges.
from scipy.optimize import curve_fit
from tools import ati_model as A
from tools.ati_model import cosine_law

d = load_replicate_table()

rows = []
for rock in ("Augen gneiss", "Psammitic schist"):
    sub = d[d.Rock_type == rock].dropna(subset=["Angle", "Tensile_strength_Mpa"])
    fit = A.fit(sub)
    e_lo, e_hi, *_ = A.profile_interval(fit, "eta", 0.0, 6.0, 0.05)
    b_lo, b_hi, *_ = A.profile_interval(fit, "beta_peak", 45.0, 89.0, 1.0)
    x, y = sub.Angle.to_numpy(float), sub.Tensile_strength_Mpa.to_numpy(float)
    p, _ = curve_fit(cosine_law, x, y, p0=[y.mean(), -1.0, 70.0], maxfev=200000)
    dense = np.linspace(0, 90, 4001)
    rows.append(dict(lithology=rock, eta=fit["eta"], eta_lo=e_lo, eta_hi=e_hi,
                     beta_p=fit["beta_peak_deg"], beta_lo=b_lo, beta_hi=b_hi,
                     R2=fit["R2"], chi2_red=fit["chi2_red"],
                     envelope_min=fit["model_min_deg"],
                     cosine_min=dense[np.argmin(cosine_law(dense, *p))]))
env = pd.DataFrame(rows)
print(env.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
for _, q in env.iterrows():
    print(f"{q.lithology:17s} envelope minimum {q.envelope_min:5.1f} deg, "
          f"cosine law {q.cosine_min:5.1f} deg, measured 75 deg")

       lithology   eta  eta_lo  eta_hi  beta_p  beta_lo  beta_hi    R2  chi2_red  envelope_min  cosine_min
    Augen gneiss 0.193   0.150   0.300  72.597   66.000   76.000 0.887     1.071        75.285      85.410
Psammitic schist 1.701   0.850   1.700  83.124   81.000   84.000 0.915     1.632        83.228      86.647

Augen gneiss      envelope minimum  75.3 deg, cosine law  85.4 deg, measured 75 deg
Psammitic schist  envelope minimum  83.2 deg, cosine law  86.6 deg, measured 75 deg


In [8]:
run("make_envelope_fit_figure")
run("make_weakening_figure")
run("make_predicted_vs_actual")
run("make_ati_supplement")
run("make_strength_by_angle_table")
run("make_strength_anisotropy_figures")

Augen gneiss      R2=0.887  chi2red=1.07  eta=0.193  beta_p=72.6  (400/400 bootstrap fits converged)
  Psammitic schist  R2=0.915  chi2red=1.63  eta=1.701  beta_p=83.1  (400/400 bootstrap fits converged)

  written: Anisotropic_model_smooth_endpoint_both_rocks.pdf


**Anisotropic_model_smooth_endpoint_both_rocks.pdf**

<IPython.core.display.Image object>

Augen gneiss      W_min = 0.915 at beta_p = 72.6 deg (eta = 0.193);  sigma/sigma_0 minimum 0.711 at 75.2 deg
  Psammitic schist  W_min = 0.608 at beta_p = 83.1 deg (eta = 1.701);  sigma/sigma_0 minimum 0.320 at 83.2 deg

  written: Smooth_weakening_factor_both_rocks.pdf


**Smooth_weakening_factor_both_rocks.pdf**

...

**combined_strength_anisotropy.pdf**

<IPython.core.display.Image object>

**ucs_vs_loading_angle_with_ci.pdf**

<IPython.core.display.Image object>

[PosixPath('manuscript/combined_strength_anisotropy.pdf'),
 PosixPath('manuscript/ucs_vs_loading_angle_with_ci.pdf')]

In [9]:
# Tractions resolved on the fabric: where the orientation dependence lives
tr = ft.traction_table()
for rock, g in tr.groupby("rock"):
    g = g.sort_values("angle_deg")
    print(rock)
    print("   sigma_n  " + "  ".join(f"{a:.0f}d={v:+6.2f}" for a, v in
                                     zip(g.angle_deg, g.sigma_n_mean_MPa)))
    print("   |tau|    " + "  ".join(f"{a:.0f}d={v:6.2f}" for a, v in
                                     zip(g.angle_deg, g.tau_abs_mean_MPa)))
rot = ft.principal_rotation_table()
for rock, g in rot.groupby("rock"):
    print(f"{rock:17s} principal rotation at most {g.rotation_median_deg.max():.2f} deg")
run("make_stress_field_figures")
# Both of these write panels for the two rocks in a single pass, so they run
# here rather than once in each lithology notebook.
run("make_stress_profile_figures")
run("make_energy_localization_figure")
run("make_matrix_shear_anchoring")
# Writes the fabric-traction and energy-localization supplementary tables,
# and brings the captions that describe these fields into line with them.
run("patch_stress_field_text")

Augen gneiss
   sigma_n  0d=-23.36  15d=-20.67  30d=-15.72  45d= -9.05  60d= -3.08  75d= +0.31  90d= +1.55
   |tau|    0d=  5.27  15d=  7.62  30d= 10.86  45d= 11.50  60d=  8.66  75d=  5.51  90d=  3.52
Psammitic schist
   sigma_n  0d=-21.92  15d=-17.82  30d=-13.30  45d= -6.73  60d= -2.01  75d= +0.32  90d= +1.13
   |tau|    0d=  3.66  15d=  5.95  30d=  8.93  45d=  8.71  60d=  6.52  75d=  2.70  90d=  2.05
Augen gneiss      principal rotation at most 0.96 deg
Psammitic schist  principal rotation at most 2.42 deg


common glyph reference (95th percentile of |sigma| over all 14 specimens) = 36.15 MPa
  wrote augen_gneiss_stress_tensors.pdf
  wrote psammitic_schist_stress_tensors.pdf

  foliation-resolved tractions (disk core):
    Augen gneiss:
      sigma_n  0d=-23.36  15d=-20.67  30d=-15.72  45d=-9.05  60d=-3.08  75d=+0.31  90d=+1.55
      |tau|    0d=5.27  15d=7.62  30d=10.86  45d=11.50  60d=8.66  75d=5.51  90d=3.52
    Psammitic schist:
      sigma_n  0d=-21.92  15d=-17.82  30d=-13.30  45d=-6.73  60d=-2.01  75d=+0.32  90d=+1.13
      |tau|    0d=3.66  15d=5.95  30d=8.93  45d=8.71  60d=6.52  75d=2.70  90d=2.05

  principal-axis rotation vs the 0 degree specimen (median, deg):
    Augen gneiss: max 0.96
    Psammitic schist: max 2.42

  written: figures + fabric_tractions, principal_rotation,
           principal_heterogeneity, orientation_gradient,
           displacement_corridor (outputs/tables/*.csv)


**augen_gneiss_stress_tensors.pdf**

<IPython.core.display.Image object>

**fabric_traction_normal.pdf**

...

**fig_S_energy_localization_psammitic_schist.pdf**

<IPython.core.display.Image object>

measured cohesion reproduces 81-112% of the cohesion each specimen's own UCS implies
  matrix tensile, adopted     : max 0.008 (specimen 12)
  matrix tensile, re-anchored : max 0.054 (specimen 12)
  wrote outputs/tables/matrix_shear_anchoring.csv


direction-circle caption: regenerated with this run's numbers
  stress-tensor caption: regenerated with this run's numbers
  fabric-traction caption: regenerated with this run's numbers
  subsec9 traction paragraph: regenerated with this run's numbers
  captions replaced, fabric-traction figure and results text inserted
  four owned blocks stamped with the edit-the-script notice
  wrote manuscript/tables/table_S_fabric_tractions.txt


[]

In [10]:
# Cross-lithology strain partitioning: needs both rocks, so it belongs here
# rather than being repeated in each lithology notebook.
from tools import strain_partitioning, plotting

both = strain_partitioning.partition_all()
contrast = strain_partitioning.lithology_contrast(both)
print('gneiss vs schist — 0 identical rows means the rocks are distinguished')
display(contrast.round(4))

paths = plotting.strain_partition_two_rocks(
    both, 'outputs/figures/strain_partitioning_two_rocks')
print('written:'); [print('  ', p) for p in paths]

gneiss vs schist — 0 identical rows means the rocks are distinguished


,Augen gneiss,Psammitic schist,difference,identical,empty_in_both
WT_pct,4.3053,5.6758,-1.3705,False,False
WS_pct,9.9687,9.6042,0.3645,False,False
MT_pct,0.0000,0.5463,-0.5463,False,False
MS_pct,44.2405,47.2883,-3.0477,False,False
below_threshold_pct,41.4854,36.8854,4.6000,False,False
weak_plane_pct,14.2740,15.2800,-1.0060,False,False
matrix_pct,44.2405,47.8346,-3.5941,False,False
tensile_pct,4.3053,6.2221,-1.9168,False,False
shear_pct,54.2093,56.8925,-2.6832,False,False
total_U_MJ_m3,121.3330,131.1684,-9.8354,False,False


written:
   outputs/figures/strain_partitioning_two_rocks.pdf
   outputs/figures/strain_partitioning_two_rocks.png


[None, None]

In [11]:
# Failure statistics, counted from the fields rather than carried as literals
import importlib.util as ilu
spec = ilu.spec_from_file_location("mfs", REPO / "scripts/make_failure_statistics_figures.py")
mfs = ilu.module_from_spec(spec); spec.loader.exec_module(mfs)
cnt = mfs.counts()
assert cnt.total.nunique() == 1, "point totals must not vary with orientation"
for rock, g in cnt.groupby("rock"):
    g = g.sort_values("angle_deg")
    print(f"{rock:17s} failed " + "  ".join(f"{a:.0f}d={v:.0f}%" for a, v in
                                            zip(g.angle_deg, g.p_fail)))
run("make_failure_statistics_figures")

Augen gneiss      failed 0d=35%  15d=40%  30d=57%  45d=49%  60d=47%  75d=14%  90d=15%


Psammitic schist  failed 0d=40%  15d=39%  30d=61%  45d=63%  60d=40%  75d=11%  90d=14%


Augen gneiss
     p_fail  0d=35%  15d=40%  30d=57%  45d=49%  60d=47%  75d=14%  90d=15%
  Psammitic schist
     p_fail  0d=40%  15d=39%  30d=61%  45d=63%  60d=40%  75d=11%  90d=14%
  Augen gneiss four-class peaks: WT 0.144 at 90deg, WS 0.215 at 60deg, MT 0.000 at 0deg, MS 0.479 at 30deg
  Psammitic schist four-class peaks: WT 0.151 at 90deg, WS 0.208 at 45deg, MT 0.008 at 60deg, MS 0.494 at 30deg

  written: conditional_failure_modes_vs_theta.pdf, pfailure_vs_theta.pdf


**conditional_failure_modes_vs_theta.pdf**

<IPython.core.display.Image object>

**pfailure_vs_theta.pdf**

<IPython.core.display.Image object>

[PosixPath('manuscript/conditional_failure_modes_vs_theta.pdf'),
 PosixPath('manuscript/pfailure_vs_theta.pdf')]

In [12]:
# Sensitivity: spacing acts on resistance only, disorder on the thresholds
from tools import sensitivity_sweeps as sw
for name, df, col in (("spacing", sw.spacing_sweep(), "spacing_m"),
                      ("disorder", sw.heterogeneity_sweep(), "heterogeneity")):
    print(name)
    for rock, g in df.groupby("rock"):
        m = g.groupby(col).delta_failure_pct.mean()
        print(f"   {rock:17s} " + "  ".join(f"{k:g}:{v:+.1f}%" for k, v in m.items()))
run("make_sensitivity_figures")

spacing
   Augen gneiss      0:+0.0%  0.001:+7.6%  0.002:+7.5%  0.003:+7.7%  0.004:+7.7%
   Psammitic schist  0:+0.0%  0.001:+7.2%  0.002:+7.1%  0.003:+7.3%  0.004:+7.4%
disorder
   Augen gneiss      0:+0.0%  0.1:+2.6%  0.2:+5.9%  0.3:+9.2%  0.4:+12.7%
   Psammitic schist  0:+0.0%  0.1:+2.3%  0.2:+5.0%  0.3:+8.1%  0.4:+11.1%


spacing:
    Augen gneiss      0:+0.0%  0.001:+7.6%  0.002:+7.5%  0.003:+7.7%  0.004:+7.7%
    Psammitic schist  0:+0.0%  0.001:+7.2%  0.002:+7.1%  0.003:+7.3%  0.004:+7.4%
  heterogeneity:
    Augen gneiss      0:+0.0%  0.1:+2.6%  0.2:+5.9%  0.3:+9.2%  0.4:+12.7%
    Psammitic schist  0:+0.0%  0.1:+2.3%  0.2:+5.0%  0.3:+8.1%  0.4:+11.1%
  max |gneiss - schist| in spacing: 0.4 percentage points
  max |gneiss - schist| in heterogeneity: 1.6 percentage points

  written: delta_failure_vs_spacing.pdf, delta_failure_vs_heterogeneity.pdf


**delta_failure_vs_heterogeneity.pdf**

<IPython.core.display.Image object>

**delta_failure_vs_spacing.pdf**

<IPython.core.display.Image object>

[PosixPath('manuscript/delta_failure_vs_heterogeneity.pdf'),
 PosixPath('manuscript/delta_failure_vs_spacing.pdf')]

In [13]:
# Softening cap: smallest value within 2% of the peak mixed-mode fraction
from tools import softening_selection as ss
print(ss.select_all().to_string(index=False, float_format=lambda v: f"{v:.4f}"))

       lithology  kC_max  kT_max  peak_metric  metric_at_selection  hit_upper_bound
    Augen gneiss  0.9000  0.3150       0.0853               0.0853             True
Psammitic schist  0.1000  0.0350       0.1495               0.1481            False


In [14]:
# Fracture traces against a loading-direction null predictor
from scipy import stats
from tools import traces as tr_mod
for frac in (0.5, 1.0):
    rows = tr_mod.compare_all(frac=frac)
    ag = tr_mod.aggregate_statistics(rows); nl = tr_mod.null_model_statistics(rows)
    df = pd.DataFrame(rows)
    r, p = stats.pearsonr(df.observed_orientation_deg, df.predicted_orientation_deg)
    print(f"window {frac:.1f}R  model MAE {ag['mae_deg']:.2f}  null MAE {nl['mae_deg']:.2f}  "
          f"null better: {nl['mae_deg'] < ag['mae_deg']}  r={r:+.2f} (p={p:.2f})")

window 0.5R  model MAE 3.35  null MAE 3.65  null better: False  r=+0.22 (p=0.45)


window 1.0R  model MAE 3.35  null MAE 3.65  null better: False  r=+0.22 (p=0.45)


In [15]:
# Supplementary bounds and convergence
run("make_eshelby_supplement")
run("make_mesh_sensitivity_figure")
run("make_fig20_supplement")   # fig_S_GGc_monotonicity + its two tables

remote stress from the gneiss disc core: sxx=3.12, syy=-23.36, txy=-0.00 MPa

  concentration factors: 1.12 to 1.73
  perturbation falls below 10% beyond 3.16 inclusion radii
  effective stiffening spans 1.04 to 1.44

  written: fig_S_eshelby_bound.pdf, tables/table_S_eshelby_bound.txt, outputs/tables/*.csv


**fig_S_eshelby_bound.pdf**

<IPython.core.display.Image object>

boundary-collocation convergence (14 specimens x 4 levels):
    specimen  1 done
    specimen  2 done
    specimen  3 done
    specimen  4 done
    specimen  5 done
    specimen  6 done
    specimen  7 done
    specimen  8 done
    specimen  9 done
    specimen 10 done
    specimen 11 done
    specimen 12 done
    specimen 13 done
    specimen 14 done
  series-order convergence (14 specimens x 8 orders):
    specimen  1 done
    specimen  2 done
    specimen  3 done
    specimen  4 done
    specimen  5 done
    specimen  6 done
    specimen  7 done
    specimen  8 done
    specimen  9 done
    specimen 10 done
    specimen 11 done
    specimen 12 done
    specimen 13 done
    specimen 14 done
    at the adopted M=48: core within 0.270% of M=96 (2.241% over the whole disc), boundary residual 6.27e-03

  at the production setting (420 collocation points) the largest deviation
  from the refined solution is 0.014% over all fourteen specimens
  changing the grid from 20081 to 31417 points 

**mesh_sensitivity.pdf**

...

**fig_S_GGc_monotonicity.pdf**

<IPython.core.display.Image object>

**fig_S_GGc_monotonicity.png**

<IPython.core.display.Image object>

[PosixPath('manuscript/fig_S_GGc_monotonicity.pdf'),
 PosixPath('manuscript/fig_S_GGc_monotonicity.png')]

In [16]:
# The supplementary tables above are written to manuscript/tables/*.txt and
# folded into manscript_revision_001.tex between markers, so the paper stays
# one .tex file for the submission portal while the tables stay regenerable.
# Table B.12 and Table B.13 are written from the frozen datasets rather
# than maintained by hand, so they cannot drift from the analysis again.
run("make_trace_and_connectivity_tables")
# Domain-sensitivity and mechanism-robustness checks reported in 4.9 and 5.5.
run("make_mechanism_robustness")
# Fitting-radius sensitivity for the trace comparison, reported in 4.9:
# the model-null comparison repeated from 0.70R to the full trace.
run("make_trace_radius_sweep")
run("inline_tables")

# Every graphic the manuscript includes must exist
import re
tex = (REPO / "manuscript/manscript_revision_001.tex").read_text(encoding="utf-8")
graphics = sorted({m.group(1) for m in
                   re.finditer(r"includegraphics\[[^\]]*\]\{([\w./-]+)\}", tex)})
missing = [g for g in graphics
           if not (REPO / "manuscript" /
                   (g if "." in Path(g).name else g + ".pdf")).exists()]
print(f"{len(graphics)} graphics included, {len(missing)} missing")
assert not missing, missing
print("every graphic the manuscript includes is present")

written: table_S_trace_validation.txt
  written: table_S_energy_localization.txt


written: trace_roughness.csv
  model straighter in 13/14 specimens, paired p = 0.0043
  written: orientation_signal_test.csv, mechanism_robustness_sweep.csv
  deviation vs alpha: r = -0.037, p = 0.899
  ordering preserved in all 12 proxy combinations: True


written: trace_radius_sweep.csv
 radius_frac  mae_deg  rmse_deg  median_deg  max_deg  n_within_5  n_within_10  null_mae_deg  model_minus_null_deg  ci_low_deg  ci_high_deg  p_paired_t  p_wilcoxon  mean_fit_se_deg  mean_points_retained
        1.00    3.535     4.881       2.059   12.913          11           13         3.373                 0.162      -0.571        0.959       0.699       0.583            0.796                 0.998
        0.95    3.414     4.314       2.230    9.597          12           14         3.455                -0.041      -0.916        0.510       0.913       0.670            0.865                 0.915
        0.90    3.517     4.440       3.377    9.908          11           14         3.690                -0.172      -1.212        0.436       0.683       0.715            0.947                 0.857
        0.85    3.347     4.253       3.285    8.754          11           14         3.651                -0.304      -1.530        0.400       0.540       1.0